# Unit vector list

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

def create_unit_vector_sga_array(time_range, dt_sec=0.5):
    """
    指定したtime_rangeの間で、0.5秒ごとに単位ベクトル(x, y, z)を作成する。
    time_range: tuple (start_time, end_time) [np.datetime64 or str]
    dt_sec: サンプリング間隔 [秒]
    """
    t_start, t_end = [np.datetime64(t) for t in time_range]

    # 時間配列を生成
    times = np.arange(t_start, t_end + np.timedelta64(int(dt_sec*1e9), 'ns'),
                      np.timedelta64(int(dt_sec*1e9), 'ns'))

    # ここでは例として、x方向→y方向→z方向を順に循環する単位ベクトルを作る
    n = len(times)
    unit_vectors = np.zeros((n, 3))
    for i in range(n):
        axis = i % 3  # 0:x, 1:y, 2:z
        unit_vectors[i, axis] = 1.0

    # xarray化
    da = xr.DataArray(
        unit_vectors,
        coords={'time': times, 'xyz': ['x', 'y', 'z']},
        dims=['time', 'xyz'],
        name='unit_vector_sga'
    )

    return da

# 使用例
time_range = ('2017-11-15T00:00:00', '2017-11-16T00:00:00')
unit_vector_sga_array = create_unit_vector_sga_array(time_range, dt_sec=0.1)
print(unit_vector_sga_array)
print(unit_vector_sga_array.time)


# sga2sgi

In [ ]:
import pyspedas as psp
import pytplot as pt

pt.store_data('unit_vector_sga', data={'x': unit_vector_sga_array.time, 'y': unit_vector_sga_array.data})

print(pt.data_quants['unit_vector_sga'])

psp.projects.erg.sga2sgi(name_in='unit_vector_sga', name_out='unit_vector_sgi')

unit_vector_sgi_array = xr.DataArray(
    pt.data_quants['unit_vector_sgi'].data,
    dims=['time', 'xyz'],
    coords={
        'time': pt.data_quants['unit_vector_sgi'].time,
        'xyz': ['x', 'y', 'z']
    },
    name='unit_vector_sgi'
)

dot = (unit_vector_sgi_array * unit_vector_sgi_array).sum(dim='xyz')
print(dot)
print(np.nanmin(dot), np.nanmax(dot), np.nanmean(dot))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
dot.plot(ax=ax)
ax.set_xlabel("Time")
ax.set_ylabel("Data")
ax.set_title("dot(unit_vector_sgi) vs time")
plt.show()

# Normalization

In [ ]:
unit_vector_sgi_array_normalized = unit_vector_sgi_array / np.sqrt(dot)
dot_sgi_normalized = (unit_vector_sgi_array_normalized * unit_vector_sgi_array_normalized).sum(dim='xyz')
print(dot_sgi_normalized)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
dot_sgi_normalized.plot(ax=ax)
ax.set_xlabel("Time")
ax.set_ylabel("Data")
ax.set_title("dot(unit_vector_sgi_normalized) vs time")
plt.show()

# sgi2dsi

In [ ]:
import pyspedas as psp
import pytplot as pt

pt.store_data('unit_vector_sgi_normalized', data={'x': unit_vector_sgi_array_normalized.time, 'y': unit_vector_sgi_array_normalized.data})

psp.projects.erg.sgi2dsi(name_in='unit_vector_sgi_normalized', name_out='unit_vector_dsi')

unit_vector_dsi_array = xr.DataArray(
    pt.data_quants['unit_vector_dsi'].data,
    dims=['time', 'xyz'],
    coords={
        'time': pt.data_quants['unit_vector_dsi'].time,
        'xyz': ['x', 'y', 'z']
    },
    name='unit_vector_dsi'
)

dot_dsi = (unit_vector_dsi_array * unit_vector_dsi_array).sum(dim='xyz')
print(dot_dsi)
print(np.nanmin(dot_dsi), np.nanmax(dot_dsi), np.nanmean(dot_dsi))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
dot_dsi.plot(ax=ax)
ax.set_xlabel("Time")
ax.set_ylabel("Data")
ax.set_title("dot(unit_vector_dsi) vs time")
plt.show()